# Walking network and nearest childcare centre for each HDB

Run the cells in order. This separate notebook reads the existing HDB and childcare files, creates a Singapore walking network, and maps the shortest walking distance to a centre for every residential HDB footprint.

**Why a graph instead of the original OSRM table?** OSRM normally optimises a profile's routing weight (often travel time); the distance of that route is not necessarily the shortest distance. Here Dijkstra's algorithm minimises edge length in metres, considering every eligible centre at once. No Docker server is required.

Outputs are saved to `output/nearest_centre_walking/`: a network map, an interactive HDB distance/route map, CSV results, GeoJSON points and routes, and a cached network.

**Interpretation:** distances are estimates on OpenStreetMap's walking network. HDB representative points and centre points are snapped to graph nodes. Straight-line access connectors at both ends are included and reported separately; they are not verified pedestrian entrances. Points over the configurable snapping limit are flagged, not given a fabricated route. “Nearest” means nearest among centres passing this check, on this graph. The childcare file is `ChildCareServices_2021.geojson`; verify its currency before interpreting present-day accessibility.


In [ ]:
# Run once if these dependencies are missing, then restart the kernel.
# %pip install "osmnx>=2,<3" geopandas networkx folium matplotlib scipy nbformat


In [ ]:
from pathlib import Path
import html
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
import folium
from branca.colormap import LinearColormap
from shapely.geometry import LineString, MultiLineString
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'output/hdb_blocks.gpkg').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository or its walking_network_analysis folder.')
OUT = ROOT / 'output/nearest_centre_walking'
OUT.mkdir(parents=True, exist_ok=True)
METRIC_CRS = 'EPSG:3414'  # Singapore SVY21, metres
RESIDENTIAL_ONLY = True
MAX_SNAP_M = 250  # review this assumption; both HDBs and centres are checked
NETWORK_BUFFER_M = 1000
GRAPH_FILE = OUT / 'singapore_walk_unsimplified.graphml'
ox.settings.use_cache = True
ox.settings.cache_folder = str(OUT / 'overpass_cache')
ox.settings.requests_timeout = 300


## 1. Load origins, centres and study boundary

Use residential HDB footprints by default (exclude carparks, markets and pavilions). Set `RESIDENTIAL_ONLY = False` to include all input footprints. Each source `OBJECTID` remains in the results. Representative points stay inside the building polygons. Invalid polygon geometries are repaired before computing points; missing or empty geometries stop execution for review.


In [ ]:
hdb = gpd.read_file(ROOT / 'output/hdb_blocks.gpkg', layer='hdb_blocks')
centres = gpd.read_file(ROOT / 'data/ChildCareServices_2021.geojson')
subzones = gpd.read_file(ROOT / 'data/MasterPlanSubZones_2019.geojson')
if RESIDENTIAL_ONLY:
    hdb = hdb.loc[hdb['is_residential'].fillna(False).astype(bool)].copy()
for label, frame in [('HDBs', hdb), ('centres', centres), ('subzones', subzones)]:
    if frame.empty or frame.crs is None:
        raise ValueError(f'{label}: empty dataset or missing CRS.')
    invalid = ~frame.geometry.is_valid & frame.geometry.notna()
    if invalid.any():
        print(f'{label}: repairing {invalid.sum()} invalid geometries')
        frame.loc[invalid, 'geometry'] = frame.loc[invalid, 'geometry'].make_valid()
    if (frame.geometry.isna() | frame.geometry.is_empty | ~frame.geometry.is_valid).any():
        raise ValueError(f'{label}: repair missing/empty/invalid geometries before continuing.')
hdb = hdb.to_crs(METRIC_CRS).reset_index(drop=True)
centres = centres.to_crs(METRIC_CRS).reset_index(drop=True)
hdb['geometry'] = hdb.geometry.representative_point()
centres['geometry'] = centres.geometry.representative_point()
hdb['hdb_id'] = hdb['OBJECTID'].astype(str)
centres['centre_id'] = centres['OBJECTID'].astype(str)
assert hdb.hdb_id.is_unique and centres.centre_id.is_unique
boundary = subzones.to_crs(METRIC_CRS).geometry.union_all().buffer(NETWORK_BUFFER_M)
# Include all input locations even if they fall just beyond the subzone boundary.
for frame in [hdb, centres]:
    outside = frame.loc[~frame.geometry.covered_by(boundary)]
    if not outside.empty:
        boundary = boundary.union(outside.geometry.buffer(NETWORK_BUFFER_M).union_all())
query_polygon = gpd.GeoSeries([boundary], crs=METRIC_CRS).to_crs(4326).iloc[0]
print(f'{len(hdb):,} HDB origins; {len(centres):,} childcare centres')


## 2. Create and map the walking network

The first run downloads OpenStreetMap walking paths through Overpass and can take several minutes for Singapore. Later runs reuse the graph file. Remove that cache to refresh OSM data or after changing the study boundary. All disconnected components are retained so unreachable buildings remain visible. Unsimplified nodes improve snapping along long streets. The download requires internet access; map background tiles also require internet.

API reference: [OSMnx documentation](https://osmnx.readthedocs.io/en/stable/user-reference.html).


In [ ]:
if GRAPH_FILE.exists():
    G = ox.load_graphml(GRAPH_FILE)
else:
    G = ox.graph_from_polygon(query_polygon, network_type='walk',
                              simplify=False, retain_all=True, truncate_by_edge=True)
    ox.save_graphml(G, GRAPH_FILE)
G = ox.project_graph(G, to_crs=METRIC_CRS)
if not G or G.number_of_edges() == 0:
    raise ValueError('The walking graph is empty.')
for u, v, k, data in G.edges(keys=True, data=True):
    if not np.isfinite(data['length']) or data['length'] < 0:
        raise ValueError(f'Invalid edge length: {(u, v, k)}')
nodes, edges = ox.graph_to_gdfs(G)
# Merge lines for a more compact display; routing still uses the full directed graph.
network_geometry = gpd.GeoSeries([edges.geometry.union_all()], crs=METRIC_CRS).to_crs(4326)
centre_latlon = hdb.to_crs(4326).geometry
map_location = [centre_latlon.y.mean(), centre_latlon.x.mean()]

def add_network(map_object, show=True):
    folium.GeoJson(network_geometry.__geo_interface__, name='Walking network (OSM)',
                   show=show, style_function=lambda _: {'color': '#64748b', 'weight': 1, 'opacity': 0.5}).add_to(map_object)

network_map = folium.Map(location=map_location, zoom_start=11, tiles='CartoDB positron', prefer_canvas=True)
add_network(network_map)
folium.LayerControl().add_to(network_map)
network_map.save(str(OUT / 'walking_network_map.html'))
print(f'{len(G):,} nodes; {G.number_of_edges():,} directed edges')
display(network_map)


## 3. Find the nearest centre by walking distance

A synthetic source connects to each eligible centre on the **reversed** graph, with centre access distance as its edge weight. One Dijkstra search therefore finds the nearest centre for every node, respecting original edge directions. Adding each HDB's fixed access distance gives its total estimate. This considers all eligible centres, rather than a shortlist chosen by straight-line distance. Equal-cost ties select one centre deterministically from input order.


In [ ]:
for frame in [hdb, centres]:
    frame['network_node'], frame['snap_m'] = ox.distance.nearest_nodes(
        G, X=frame.geometry.x.to_numpy(), Y=frame.geometry.y.to_numpy(), return_dist=True)
centres['eligible'] = centres.snap_m <= MAX_SNAP_M
centres.drop(columns='geometry').to_csv(OUT / 'centre_snap_review.csv', index=False)
eligible = centres.loc[centres.eligible].copy()
if eligible.empty:
    raise ValueError('No centres pass the snapping check. Review centre_snap_review.csv.')
print(f'{len(eligible):,} eligible centres; {(~centres.eligible).sum():,} excluded by snapping limit')

def nearest_centre_tree(graph, candidates):
    reverse = graph.reverse(copy=True)
    source = '__all_centres__'
    owners = {}
    # At a shared node, the smallest centre connector wins.
    for row in candidates.sort_values('snap_m', kind='stable').itertuples():
        node = int(row.network_node)
        if node not in owners:
            owners[node] = row.Index
            reverse.add_edge(source, node, length=float(row.snap_m))
    predecessor, distance = nx.dijkstra_predecessor_and_distance(reverse, source, weight='length')
    return source, owners, predecessor, distance

source, owners, predecessor, distance = nearest_centre_tree(G, eligible)

def trace_to_centre(start):
    path = [int(start)]
    while predecessor[path[-1]][0] != source:
        path.append(predecessor[path[-1]][0])
    return path, owners[path[-1]]

records, route_records = [], []
for row in hdb.itertuples():
    node = int(row.network_node)
    record = dict(hdb_id=row.hdb_id, hdb_snap_m=float(row.snap_m),
                  nearest_centre_id=None, nearest_centre_name=None,
                  centre_snap_m=np.nan, network_distance_m=np.nan, walking_distance_m=np.nan,
                  status='snap_too_far' if row.snap_m > MAX_SNAP_M else 'unreachable')
    if row.snap_m <= MAX_SNAP_M and node in distance:
        path, centre_index = trace_to_centre(node)
        centre = centres.loc[centre_index]
        total = float(distance[node] + row.snap_m)
        record.update(nearest_centre_id=centre.centre_id, nearest_centre_name=centre.NAME,
                      centre_snap_m=float(centre.snap_m),
                      network_distance_m=max(0.0, float(distance[node] - centre.snap_m)),
                      walking_distance_m=total, status='ok')
        # Store access connectors separately from actual network edges.
        parts = []
        for u, v in zip(path[:-1], path[1:]):
            edge = min(G[u][v].values(), key=lambda attr: attr['length'])
            parts.append(edge.get('geometry', LineString([(G.nodes[u]['x'], G.nodes[u]['y']),
                                                          (G.nodes[v]['x'], G.nodes[v]['y'])])))
        if parts:
            route_records.append(dict(hdb_id=row.hdb_id, centre_id=centre.centre_id,
                                      walking_distance_m=total, segment='network', geometry=MultiLineString(parts)))
        for point, endpoint in [(row.geometry, path[0]), (centre.geometry, path[-1])]:
            xy = (G.nodes[endpoint]['x'], G.nodes[endpoint]['y'])
            if point.distance(nodes.loc[endpoint].geometry) > 0:
                route_records.append(dict(hdb_id=row.hdb_id, centre_id=centre.centre_id,
                                          walking_distance_m=total, segment='estimated access',
                                          geometry=LineString([(point.x, point.y), xy])))
    records.append(record)
results = hdb.merge(pd.DataFrame(records), on='hdb_id', validate='one_to_one')
results['walking_distance_km'] = results.walking_distance_m / 1000
routes = gpd.GeoDataFrame(route_records, columns=['hdb_id', 'centre_id', 'walking_distance_m', 'segment', 'geometry'], crs=METRIC_CRS)
assert len(results) == len(hdb)
assert results.loc[results.status.eq('ok'), 'walking_distance_m'].ge(0).all()
display(results.status.value_counts().rename_axis('status').to_frame('HDB count'))
display(results.walking_distance_m.describe().to_frame('Walking distance (m)'))


## 4. Map HDB distances and routes, and export results

Click an HDB for its selected centre and distance. Grey HDBs need review. Toggle the network, route, estimated access and centre layers independently. All routes are included; they start hidden to keep the initial map readable. Dashed access lines are estimates, not mapped footpaths. The complete map can be large for a nationwide dataset.


In [ ]:
results.drop(columns='geometry').to_csv(OUT / 'hdb_nearest_centre.csv', index=False)
point_columns = ['hdb_id', 'nearest_centre_id', 'nearest_centre_name', 'walking_distance_m',
                 'walking_distance_km', 'network_distance_m', 'hdb_snap_m', 'centre_snap_m', 'status', 'geometry']
points_wgs = results[point_columns].to_crs(4326)
points_wgs.to_file(OUT / 'hdb_nearest_centre.geojson', driver='GeoJSON')
if not routes.empty:
    routes.to_crs(4326).to_file(OUT / 'nearest_centre_routes.geojson', driver='GeoJSON')

valid_distances = results.walking_distance_m.dropna()
maximum = max(1.0, float(valid_distances.max())) if len(valid_distances) else 1.0
colours = LinearColormap(['#15803d', '#facc15', '#ea580c', '#991b1b'], vmin=0, vmax=maximum,
                        caption='Estimated walking distance to nearest centre (metres)')
distance_map = folium.Map(location=map_location, zoom_start=11, tiles='CartoDB positron', prefer_canvas=True)
add_network(distance_map, show=False)
for segment in ['network', 'estimated access']:
    subset = routes.loc[routes.segment.eq(segment)]
    if not subset.empty:
        style = {'color': '#2563eb', 'weight': 2, 'opacity': 0.6}
        if segment == 'estimated access':
            style.update(color='#9333ea', dashArray='5,5')
        folium.GeoJson(subset.to_crs(4326).to_json(), name=f'Nearest-centre routes: {segment}', show=False,
                       style_function=lambda _, s=style: s,
                       tooltip=folium.GeoJsonTooltip(fields=['hdb_id', 'centre_id', 'walking_distance_m'])).add_to(distance_map)
centre_layer = folium.FeatureGroup(name='Childcare centres (green eligible / grey excluded)', show=False)
for row in centres.to_crs(4326).itertuples():
    folium.CircleMarker([row.geometry.y, row.geometry.x], radius=3,
                        color='#047857' if row.eligible else '#6b7280', fill=True,
                        tooltip=html.escape(str(row.NAME))).add_to(centre_layer)
centre_layer.add_to(distance_map)

def hdb_style(feature):
    properties = feature['properties']
    colour = colours(properties['walking_distance_m']) if properties['status'] == 'ok' else '#6b7280'
    return {'color': colour, 'fillColor': colour, 'fillOpacity': 0.85, 'weight': 0.5}

folium.GeoJson(points_wgs.to_json(), name='HDB distance to nearest centre',
               marker=folium.CircleMarker(radius=4), style_function=hdb_style,
               tooltip=folium.GeoJsonTooltip(fields=['hdb_id', 'nearest_centre_name', 'walking_distance_m', 'status']),
               popup=folium.GeoJsonPopup(fields=['hdb_id', 'nearest_centre_id', 'nearest_centre_name',
                                                'walking_distance_m', 'network_distance_m', 'hdb_snap_m', 'centre_snap_m', 'status'])).add_to(distance_map)
colours.add_to(distance_map)
folium.LayerControl(collapsed=False).add_to(distance_map)
distance_map.save(str(OUT / 'hdb_nearest_centre_map.html'))
print(f'Saved maps and results to {OUT}')
display(distance_map)
